# Multi-Agent AI Grading Demo — Unit 4.1 Q6 (Geology)

This notebook demonstrates how our **5-agent AI grading ensemble** works for elementary geology responses.

## Question (Q6)
Look closely at the photo. Think about a system that has parts that work together.

**Identify two or three parts of a system and explain how they work together to cause change over time.**

## Multi-agent design
1. **Agents 1–5** — Each agent uses a different rubric prompt variant (`prompts.py`) and independently grades the same student answer (0–3).
2. **Consensus logic** — We count agent votes:
   - **Agent agree** = at least 4 of 5 agents assign the same grade
   - **Agent unanimous** = all 5 agents assign the same grade
   - **Consensus grade** = the grade with ≥4 votes (otherwise `None`)
   - **Majority vote** = the most common grade across agents

This mirrors the pipeline in `main.py`, but runs on four illustrative student answers instead of a CSV file.

## Setup

**Run the cells in order.** The next cell installs dependencies into the active Python kernel.

Requires a `.env` file in this folder:

```
GEMINI_API_KEY=your_key_here
# Optional if Google APIs are blocked on your network:
# HTTPS_PROXY=http://127.0.0.1:7890
```

This notebook uses the **Gemini REST API** (not gRPC) for better compatibility with VPN/proxy setups.

In [7]:
import subprocess
import sys
from pathlib import Path

requirements_file = Path("requirements.txt")

if requirements_file.exists():
    packages = [
        line.strip()
        for line in requirements_file.read_text(encoding="utf-8").splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]
else:
    packages = [
        "google-generativeai",
        "python-dotenv",
        "pandas",
        "ipython",
        "ipykernel",
    ]

print(f"Installing into: {sys.executable}")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages]
)
print("All requirements installed.")

Installing into: c:\Users\Gino\Desktop\coding\.conda\python.exe
All requirements installed.


In [8]:
import asyncio
import os
import re
from collections import Counter

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display

from prompts import (
    base_prompt1,
    base_prompt2,
    base_prompt3,
    base_prompt4,
    base_prompt5,
)

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Add it to the .env file in this folder.")

MODEL_NAME = "gemini-2.5-flash"
N_GRADERS = 5
GRADE_REGEX = re.compile(r"\b[0-4]\b")
REQUEST_TIMEOUT = 120
RUN_AGENTS_SEQUENTIALLY = True  # safer on slow or proxied networks

grading_prompts = [base_prompt1, base_prompt2, base_prompt3, base_prompt4, base_prompt5]

PROXIES = {}
if os.getenv("HTTPS_PROXY"):
    PROXIES["https"] = os.getenv("HTTPS_PROXY")
if os.getenv("HTTP_PROXY"):
    PROXIES["http"] = os.getenv("HTTP_PROXY")


def prompt_text(prompt_obj):
    """Extract rubric text from prompt dicts in prompts.py."""
    if isinstance(prompt_obj, dict):
        return next(iter(prompt_obj.values()))
    return prompt_obj


def test_gemini_connection():
    """Quick REST connectivity check before running all agents."""
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent"
    response = requests.post(
        url,
        params={"key": API_KEY},
        json={"contents": [{"parts": [{"text": "Reply with exactly: OK"}]}]},
        timeout=30,
        proxies=PROXIES or None,
    )
    response.raise_for_status()
    text = response.json()["candidates"][0]["content"]["parts"][0]["text"]
    return text.strip()


try:
    ping = test_gemini_connection()
    print("Gemini REST API connected.")
    print(f"Model: {MODEL_NAME}")
    print(f"Graders: {N_GRADERS}")
    print(f"Test reply: {ping[:80]}")
    if PROXIES:
        print(f"Using proxy: {PROXIES}")
except requests.RequestException as exc:
    print("Could not reach Gemini API.")
    print(f"Error: {exc}")
    print("\nIf you are in a region where Google is blocked, turn on a VPN or set HTTPS_PROXY in .env")

Gemini REST API connected.
Model: gemini-2.5-flash
Graders: 5
Test reply: OK


## Example student answers

| Example | Student answer |
|---------|----------------|
| 1 | Mountains are on moving earth plates that raise them over time, as they push against each other. |
| 2 | The river are running around and shape the terretories of the land gives the envidence of changing over time |
| 3 | I do not know |
| 4 | They change because it was bright outside, and the water dried up. |

In [9]:
EXAMPLES = [
    "Mountains are on moving earth plates that raise them over time, as they push against each other.",
    "The river are running around and shape the terretories of the land gives the envidence of changing over time",
    "I do not know",
    "They change because it was bright outside, and the water dried up.",
]

pd.DataFrame(
    {"Example": [f"Example {i}" for i in range(1, len(EXAMPLES) + 1)], "Student answer": EXAMPLES}
)

,Example,Student answer,Notes
0,Example 1 — Plate tectonics,Mountains are on moving earth plates that rais...,Names plates + mountains and explains interact...
1,Example 2 — River erosion,The river are running around and shape the ter...,Mentions river + land change; grammar is weak ...
2,Example 3 — No answer,I do not know,Should receive score 0 (no system parts or exp...
3,Example 4 — Incorrect reasoning,"They change because it was bright outside, and...",Vague 'they'; weather/sunlight is not a valid ...


## Grading engine (from `main.py`)

In [10]:
class GeminiClient:
    """Call Gemini via HTTPS REST (avoids gRPC connection issues)."""

    def __init__(self, model_name=MODEL_NAME, api_key=API_KEY, timeout=REQUEST_TIMEOUT):
        self.model_name = model_name
        self.api_key = api_key
        self.timeout = timeout
        self.url = (
            f"https://generativelanguage.googleapis.com/v1beta/models/"
            f"{self.model_name}:generateContent"
        )

    def _generate_sync(self, prompt: str) -> str:
        response = requests.post(
            self.url,
            params={"key": self.api_key},
            json={"contents": [{"parts": [{"text": prompt}]}]},
            timeout=self.timeout,
            proxies=PROXIES or None,
        )
        response.raise_for_status()
        data = response.json()
        return data["candidates"][0]["content"]["parts"][0]["text"].strip()

    async def generate(self, prompt: str) -> str:
        return await asyncio.to_thread(self._generate_sync, prompt)


def extract_grade(reply: str):
    match = GRADE_REGEX.search(reply or "")
    return int(match.group()) if match else None


def compute_consensus(agent_grades):
    counts = Counter(g for g in agent_grades if g is not None)
    if counts:
        top_grade, top_count = counts.most_common(1)[0]
    else:
        top_grade, top_count = (None, 0)

    return {
        "agent_agree": int(top_count >= 4),
        "agent_unanimous": int(top_count == 5),
        "agent_consensus_grade": top_grade if top_count >= 4 else None,
        "agent_majority_vote": top_grade if counts else None,
        "grade_counts": dict(sorted(counts.items())),
    }


async def grade_with_multi_agents(student_answer: str, verbose=True):
    """Run all 5 grading agents on one student answer."""
    gemini_client = GeminiClient()
    results = []

    for i in range(N_GRADERS):
        agent_num = i + 1
        base = prompt_text(grading_prompts[i])
        prompt = f"{base}\n\nStudent answer:\n{student_answer}"

        if verbose:
            print(f"Calling agent {agent_num}/{N_GRADERS}...")

        reply = await gemini_client.generate(prompt)
        reply = (reply or "").strip()
        grade = extract_grade(reply)
        result = {"agent": agent_num, "grade": grade, "explanation": reply}
        results.append(result)

        if verbose:
            print(f"\n--- Agent {agent_num} ---\n{reply}\n")

        if RUN_AGENTS_SEQUENTIALLY and agent_num < N_GRADERS:
            await asyncio.sleep(0.5)

    agent_grades = [r["grade"] for r in results]
    consensus = compute_consensus(agent_grades)

    record = {
        "student_answer": student_answer,
        "agent_grades": agent_grades,
        **consensus,
    }
    for r in results:
        record[f"ai_grade_agent_{r['agent']}"] = r["grade"]
        record[f"ai_explanation_agent_{r['agent']}"] = r["explanation"]

    return record

## Run the demo on all four examples

Each example is sent to all 5 agents. Agent responses are printed below, followed by a summary table.

In [11]:
async def run_demo():
    all_records = []

    for idx, answer in enumerate(EXAMPLES, start=1):
        example_label = f"Example {idx}"
        display(Markdown(f"## {example_label}"))
        display(Markdown(f"**Student answer:** {answer}"))
        print(f"Grading example {idx}/{len(EXAMPLES)}...")

        record = await grade_with_multi_agents(answer, verbose=True)
        record["example"] = example_label
        all_records.append(record)

        display(
            Markdown(
                f"**Grades:** {record['agent_grades']}  \n"
                f"**Majority vote:** {record['agent_majority_vote']}  \n"
                f"**Consensus (≥4 agree):** {record['agent_consensus_grade']}  \n"
                f"**Agents agree (≥4):** {bool(record['agent_agree'])}  \n"
                f"**Unanimous:** {bool(record['agent_unanimous'])}"
            )
        )
        print("=" * 80)

    return pd.DataFrame(all_records)


results_df = await run_demo()
results_df

## Example 1 — Plate tectonics

**Student answer:** Mountains are on moving earth plates that raise them over time, as they push against each other.

Grading example 1/4...
Calling agent 1/5...

--- Agent 1 ---
The grading from AI is **3**.

The reason for this grading is because the student clearly identifies two parts of a system: "Mountains" and "moving earth plates." They then describe how these parts work together, explaining that the plates "push against each other" to "raise them [mountains]." Finally, the student explicitly states that this process causes change "over time." This response fulfills all three criteria for Score Level 3: identifying parts, connecting them with an explanation of their interaction, and describing how the system changes over time.

Calling agent 2/5...

--- Agent 2 ---
- The grading from AI is **3**.
- The reason for this grading is because **the student identifies two parts of the system: "Mountains" and "moving earth plates." They explain how these parts work together by stating that the "earth plates... push against each other," which causes the mountains to "raise them over time." This explana

**Grades:** [3, 3, 3, 3, 3]  
**Majority vote:** 3  
**Consensus (≥4 agree):** 3  
**Agents agree (≥4):** True  
**Unanimous:** True

## Example 2 — River erosion

...

## Example 4 — Incorrect reasoning

**Student answer:** They change because it was bright outside, and the water dried up.

Grading example 4/4...
Calling agent 1/5...

--- Agent 1 ---
The grading from AI is **3**.
The reason for this grading is because the student identifies two parts of a natural system: "water" and implicitly, the "sun/brightness outside" (which falls under weather elements in the provided components). They explain how these two parts work together ("bright outside" causes the water to dry up) and clearly state a change over time ("the water dried up"). This meets all three criteria for a Score Level 3, especially considering the provided examples which show a degree of leniency for elementary-level responses.

Calling agent 2/5...

--- Agent 2 ---
- The grading from AI is **3**.
- The reason for this grading is because **the student identifies two parts of a system: "water" and implicitly "bright outside" (referring to sunlight or weather). They explain how these parts work together by stating that the "bright outside" caused the "water dried up," which describes an interaction (evapora

**Grades:** [3, 3, 2, 3, 3]  
**Majority vote:** 3  
**Consensus (≥4 agree):** 3  
**Agents agree (≥4):** True  
**Unanimous:** False

,student_answer,agent_grades,agent_agree,agent_unanimous,agent_consensus_grade,agent_majority_vote,grade_counts,ai_grade_agent_1,ai_explanation_agent_1,ai_grade_agent_2,ai_explanation_agent_2,ai_grade_agent_3,ai_explanation_agent_3,ai_grade_agent_4,ai_explanation_agent_4,ai_grade_agent_5,ai_explanation_agent_5,example
0,Mountains are on moving earth plates that rais...,"[3, 3, 3, 3, 3]",1,1,3,3,{3: 5},3,The grading from AI is **3**.\n\nThe reason fo...,3,- The grading from AI is **3**.\n- The reason ...,3,- The grading from AI is **3**.\n- The reason ...,3,- The grading from AI is **[3]**.\n- The reaso...,3,- The grading from AI is **3**.\n- The reason ...,Example 1 — Plate tectonics
1,The river are running around and shape the ter...,"[3, 3, 3, 3, 3]",1,1,3,3,{3: 5},3,- The grading from AI is **3**.\n- The reason ...,3,- The grading from AI is **3**.\n- The reason ...,3,- The grading from AI is **3**.\n- The reason ...,3,- The grading from AI is **[3]**.\n- The reaso...,3,The grading from AI is **3**.\n\nThe reason fo...,Example 2 — River erosion
2,I do not know,"[0, 0, 0, 0, 0]",1,1,0,0,{0: 5},0,- The grading from AI is **[0]**.\n- The reaso...,0,- The grading from AI is **[0]**.\n- The reaso...,0,- The grading from AI is **[0]**.\n- The reaso...,0,- The grading from AI is **[0]**.\n- The reaso...,0,The grading from AI is **0**.\nThe reason for ...,Example 3 — No answer
3,"They change because it was bright outside, and...","[3, 3, 2, 3, 3]",1,0,3,3,"{2: 1, 3: 4}",3,The grading from AI is **3**.\nThe reason for ...,3,- The grading from AI is **3**.\n- The reason ...,2,- The grading from AI is **2**.\n- The reason ...,3,- The grading from AI is **[3]**.\n- The reaso...,3,- The grading from AI is **3**.\n\n- The reaso...,Example 4 — Incorrect reasoning


## Summary table

In [12]:
summary_cols = [
    "example",
    "agent_grades",
    "agent_majority_vote",
    "agent_consensus_grade",
    "agent_agree",
    "agent_unanimous",
    "grade_counts",
]

summary = results_df[summary_cols].copy()
summary["agent_agree"] = summary["agent_agree"].map({1: "Yes", 0: "No"})
summary["agent_unanimous"] = summary["agent_unanimous"].map({1: "Yes", 0: "No"})
summary

,example,agent_grades,agent_majority_vote,agent_consensus_grade,agent_agree,agent_unanimous,grade_counts
0,Example 1 — Plate tectonics,"[3, 3, 3, 3, 3]",3,3,Yes,Yes,{3: 5}
1,Example 2 — River erosion,"[3, 3, 3, 3, 3]",3,3,Yes,Yes,{3: 5}
2,Example 3 — No answer,"[0, 0, 0, 0, 0]",0,0,Yes,Yes,{0: 5}
3,Example 4 — Incorrect reasoning,"[3, 3, 2, 3, 3]",3,3,Yes,No,"{2: 1, 3: 4}"


## Side-by-side agent grades

In [13]:
agent_grade_cols = [f"ai_grade_agent_{i}" for i in range(1, N_GRADERS + 1)]

grade_matrix = results_df[["example"] + agent_grade_cols].set_index("example")
grade_matrix.columns = [f"Agent {i}" for i in range(1, N_GRADERS + 1)]
grade_matrix

,Agent 1,Agent 2,Agent 3,Agent 4,Agent 5
example,,,,,
Example 1 — Plate tectonics,3,3,3,3,3
Example 2 — River erosion,3,3,3,3,3
Example 3 — No answer,0,0,0,0,0
Example 4 — Incorrect reasoning,3,3,2,3,3


## Inspect one agent's full explanation

Change `example_index` (0–3) and `agent_num` (1–5) to read any agent's rubric-aligned reasoning.

In [14]:
example_index = 0
agent_num = 1

row = results_df.iloc[example_index]
display(Markdown(f"### Example {example_index + 1}"))
display(Markdown(f"**Answer:** {row['student_answer']}"))
display(Markdown(f"**Agent {agent_num} grade:** {row[f'ai_grade_agent_{agent_num}']}"))
print(row[f"ai_explanation_agent_{agent_num}"])

### Example 1 — Plate tectonics

**Answer:** Mountains are on moving earth plates that raise them over time, as they push against each other.

**Agent 1 grade:** 3

The grading from AI is **3**.

The reason for this grading is because the student clearly identifies two parts of a system: "Mountains" and "moving earth plates." They then describe how these parts work together, explaining that the plates "push against each other" to "raise them [mountains]." Finally, the student explicitly states that this process causes change "over time." This response fulfills all three criteria for Score Level 3: identifying parts, connecting them with an explanation of their interaction, and describing how the system changes over time.


## How to interpret results

When agents disagree, compare their explanations in the cells above. High agreement (≥4/5) gives a reliable **consensus grade** for downstream analysis, matching the logic used in `main.py` for batch CSV grading.